# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [1]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [2]:
import torch

# ========== 1. 数据（线索 3，训练中保持不变）==========
x_data = x.float()
y_data = y.float()
n = x_data.numel()
assert x_data.shape == y_data.shape, 'x 与 y 的样本数必须一致'
print('样本数 N =', n)
print('x =', x_data.tolist())
print('y =', y_data.tolist())

# ========== 2. 三个待学习参数（线索 1）==========
torch.manual_seed(0)
w1 = torch.tensor(1.0, requires_grad=True)   # 一次项系数
w2 = torch.tensor(1.0, requires_grad=True)   # 二次项系数
w3 = torch.tensor(1.0, requires_grad=True)   # 常数项

# ========== 3. 预测函数：y_hat = w1*x + w2*x**2 + w3 ==========
def y_hat(x_input):
    return w1 * x_input + w2 * x_input ** 2 + w3

# ========== 4. 损失：L = (1/N) * sum |y_i - y_hat_i| ==========
def mae(y_true, y_pred):
    return torch.mean(torch.abs(y_true - y_pred))

# 训练前的水平，作为后面判断"有没有学到东西"的基准
with torch.no_grad():
    print()
    print(f'训练前：w1={w1.item():.6f}  w2={w2.item():.6f}  w3={w3.item():.6f}'
          f'  平均绝对误差={mae(y_data, y_hat(x_data)).item():.6f}')

# ========== 5. 优化器：三个参数一起交给它 ==========
base_lr = 0.2
decay = 0.998
epochs = 5000
optimizer = torch.optim.SGD([w1, w2, w3], lr=base_lr)
history = []

for epoch in range(1, epochs + 1):
    lr = base_lr * decay ** (epoch - 1)      # 每轮把学习率乘 0.998，后期步子变小
    optimizer.param_groups[0]['lr'] = lr
    optimizer.zero_grad()                    # 清掉上一轮的梯度（不会把参数清零）
    loss = mae(y_data, y_hat(x_data))        # 前向：算平均绝对误差
    loss.backward()                          # 反向：求三个参数的梯度
    optimizer.step()                         # 更新三个参数
    if epoch == 1 or epoch % 1000 == 0:
        with torch.no_grad():
            history.append((epoch, lr, w1.item(), w2.item(), w3.item(),
                            mae(y_data, y_hat(x_data)).item()))

print()
print('轮数     学习率        w1          w2          w3        平均绝对误差')
for epoch, lr, a, b, c, l in history:
    print(f'{epoch:5d}  {lr:.6f}  {a:9.6f}  {b:9.6f}  {c:9.6f}  {l:.6f}')

样本数 N = 9
x = [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
y = [-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0]

训练前：w1=1.000000  w2=1.000000  w3=1.000000  平均绝对误差=2.000000

轮数     学习率        w1          w2          w3        平均绝对误差
    1  0.200000   1.022222   0.711111   0.955556  1.604938
 1000  0.027067   1.477699  -0.467120   2.004416  0.059216
 2000  0.003656   1.496149  -0.496245   2.001632  0.007890
 3000  0.000494   1.499437  -0.499598   2.000238  0.000908
 4000  0.000067   1.499956  -0.499947   2.000020  0.000109
 5000  0.000009   1.499993  -0.499993   2.000007  0.000018


### 我的观察与说明

和作业 01 相比，代码主要做了几处调整:把单个学习参数 w 扩展成了 w1、w2、w3 三个参数同步更新，预测公式换成了题目要求的二次多项式，损失计算从单样本绝对误差改成了 9 个样本的平均误差，同时微调了初始学习率和训练轮数，保留学习率随训练衰减的策略，避免参数在最优值附近来回跳动。

判断这次训练有效可以从三点看:训练全程损失整体持续下降，最终稳定在接近 0 的水平；每个样本的预测值和真实值差距都很小；最终学到的参数和数据原本的二次函数系数基本一致，说明模型成功拟合了数据规律，训练是有效的。

## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。